# Machine Learning Prep: Load Reaction Databases

This notebook prepares data for the Random Forest analysis requested in the revisions. For now, we will:

- Locate and load all filtered reaction databases for multimer sizes 4 and 5.
- Provide concise previews to verify successful loading.
- Add placeholders to later ingest band intensity Excel files.

Each step is documented with a short markdown cell for clarity.

# Imports and display options

We import standard data libraries and set display preferences for quick inspection.

In [438]:
# Import Required Libraries
import sys
import warnings
from pathlib import Path
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 50)
pd.set_option("display.width", 120)

# Define paths and locate the filtered reaction databases

We resolve the repository root robustly (independent of current working directory) and point to `back_end/data/filtered_reaction_database`.

In [439]:
# Robustly find repository root and filtered data directory

def find_repo_root(start: Path) -> Path:
    """Ascend from start until we find a directory containing 'requirements.txt' and 'back_end'."""
    cur = start.resolve()
    for _ in range(10):  # guard against runaway
        if (cur / "requirements.txt").exists() and (cur / "back_end").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start.resolve()

# Start from the notebook directory
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = find_repo_root(NOTEBOOK_DIR)
FILTERED_DIR = REPO_ROOT / "back_end" / "data" / "filtered_reaction_database"

print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Repo root:     {REPO_ROOT}")
print(f"Filtered dir:  {FILTERED_DIR}")

assert FILTERED_DIR.exists(), "Expected filtered_reaction_database directory to exist."

Notebook dir: /Users/ekummelstedt/le_code_base/ubiquitinformatics/back_end/revisions
Repo root:     /Users/ekummelstedt/le_code_base/ubiquitinformatics
Filtered dir:  /Users/ekummelstedt/le_code_base/ubiquitinformatics/back_end/data/filtered_reaction_database


# Discover available datasets under filtered_reaction_database

We enumerate CSV files for each multimer size and dataset type (combined, context_history, donor_history, reaction_history, ubiquitin_history).

In [440]:
# Enumerate CSV files
from collections import defaultdict

csv_index = {}
for size_dir in sorted(FILTERED_DIR.glob("multimer_size_*")):
    size_key = size_dir.name  # e.g., 'multimer_size_4'
    for csv_path in sorted(size_dir.glob("*.csv")):
        name_key = csv_path.stem  # e.g., 'combined_database'
        key = f"{size_key}/{name_key}"
        csv_index[key] = csv_path

print(f"Discovered {len(csv_index)} CSVs:\n")
for k, p in sorted(csv_index.items()):
    print(f"- {k}: {p.relative_to(REPO_ROOT)}")

Discovered 10 CSVs:

- multimer_size_4/combined_database: back_end/data/filtered_reaction_database/multimer_size_4/combined_database.csv
- multimer_size_4/context_history: back_end/data/filtered_reaction_database/multimer_size_4/context_history.csv
- multimer_size_4/donor_history: back_end/data/filtered_reaction_database/multimer_size_4/donor_history.csv
- multimer_size_4/reaction_history: back_end/data/filtered_reaction_database/multimer_size_4/reaction_history.csv
- multimer_size_4/ubiquitin_history: back_end/data/filtered_reaction_database/multimer_size_4/ubiquitin_history.csv
- multimer_size_5/combined_database: back_end/data/filtered_reaction_database/multimer_size_5/combined_database.csv
- multimer_size_5/context_history: back_end/data/filtered_reaction_database/multimer_size_5/context_history.csv
- multimer_size_5/donor_history: back_end/data/filtered_reaction_database/multimer_size_5/donor_history.csv
- multimer_size_5/reaction_history: back_end/data/filtered_reaction_database/

# Load datasets into DataFrames

We load each discovered CSV into a `datasets` dictionary keyed by `multimer_size_x/<name>` and report shapes.

In [441]:
# Read CSVs into a dictionary of DataFrames

datasets: dict[str, pd.DataFrame] = {}
for key, path in sorted(csv_index.items()):
    try:
        df = pd.read_csv(path)
        datasets[key] = df
    except Exception as e:
        print(f"Failed to load {key} ({path.name}): {e}")

print("\nLoaded DataFrames:")
for k, df in datasets.items():
    print(f"- {k:40s} shape={df.shape}")

# Convenience: list of keys grouped by multimer size
keys_by_size = {}
for k in datasets:
    size = k.split('/')[0]
    keys_by_size.setdefault(size, []).append(k)

for size, keys in sorted(keys_by_size.items()):
    print(f"\n{size}: {len(keys)} tables -> {sorted([k.split('/')[-1] for k in keys])}")


Loaded DataFrames:
- multimer_size_4/combined_database        shape=(78, 12)
- multimer_size_4/context_history          shape=(26, 11)
- multimer_size_4/donor_history            shape=(26, 10)
- multimer_size_4/reaction_history         shape=(26, 10)
- multimer_size_4/ubiquitin_history        shape=(26, 11)
- multimer_size_5/combined_database        shape=(342, 14)
- multimer_size_5/context_history          shape=(114, 13)
- multimer_size_5/donor_history            shape=(114, 12)
- multimer_size_5/reaction_history         shape=(114, 12)
- multimer_size_5/ubiquitin_history        shape=(114, 13)

multimer_size_4: 5 tables -> ['combined_database', 'context_history', 'donor_history', 'reaction_history', 'ubiquitin_history']

multimer_size_5: 5 tables -> ['combined_database', 'context_history', 'donor_history', 'reaction_history', 'ubiquitin_history']


# Quick previews for sanity check

We show a few rows from each table to confirm the load worked and to familiarize ourselves with columns.

In [442]:
# Display small samples from each dataset
for k in sorted(datasets):
    df = datasets[k]
    print("\n==>", k, f"shape={df.shape}")
    display(df.head(3))


==> multimer_size_4/combined_database shape=(78, 12)


,Unnamed: 0,index,multimer_id,used_in_synthesis,table_origin,initial_acceptor,dimer_formation,dimer_deprotection,trimer_formation,trimer_deprotection,tetramer_formation,final_multimer
0,0,31,Ub4_4,1,Donors,NaN,ubi_ubq_1_K48_SMAC_K63_ABOC,NaN,ubi_ubq_1_K48_ABOC_K63_ABOC,NaN,ubi_ubq_1_K48_ABOC_K63_ABOC,NaN
1,1,55,Ub4_4,0,Donors,NaN,ubi_ubq_1_K48_SMAC_K63_ABOC,NaN,ubi_ubq_1_K48_ABOC_K63_ABOC,NaN,ubi_ubq_1_K48_ABOC_K63_ABOC,NaN
2,2,75,Ub4_4,0,Donors,NaN,ubi_ubq_1_K48_SMAC_K63_ABOC,NaN,ubi_ubq_1_K48_ABOC_K63_ABOC,NaN,ubi_ubq_1_K48_ABOC_K63_ABOC,NaN



==> multimer_size_4/context_history shape=(26, 11)


,Unnamed: 0,index,multimer_id,used_in_synthesis,initial_acceptor,dimer_formation,dimer_deprotection,trimer_formation,trimer_deprotection,tetramer_formation,final_multimer
0,0,31,Ub4_4,1,"{'chain_number_list': [1, 2], 'chain_length_list': [83], 'multimer_string_name': 'his-GG-1ubq-1-()', 'nomenclature_w...","{'chain_number_list': [1, 2, 3], 'chain_length_list': [83, 76], 'multimer_string_name': 'his-GG-1ubq-1-(<K48_1ubq-2-...","{'chain_number_list': [1, 2, 3], 'chain_length_list': [83, 76], 'multimer_string_name': 'his-GG-1ubq-1-(<K48_1ubq-2-...","{'chain_number_list': [1, 2, 3, 4], 'chain_length_list': [83, 76, 76], 'multimer_string_name': 'his-GG-1ubq-1-(<K48_...","{'chain_number_list': [1, 2, 3, 4], 'chain_length_list': [83, 76, 76], 'multimer_string_name': 'his-GG-1ubq-1-(<K48_...","{'chain_number_list': [1, 2, 3, 4, 5], 'chain_length_list': [83, 76, 76, 76], 'multimer_string_name': 'his-GG-1ubq-1...","{'chain_number_list': [1, 2, 3, 4, 5], 'chain_length_list': [83, 76, 76, 76], 'multimer_string_name': 'his-GG-1ubq-1..."
1,1,55,Ub4_4,0,"{'chain_number_list': [1, 2], 'chain_length_list': [83], 'multimer_string_name': 'his-GG-1ubq-1-()', 'nomenclature_w...","{'chain_number_list': [1, 2, 3], 'chain_length_list': [83, 76], 'multimer_string_name': 'his-GG-1ubq-1-(<K48_1ubq-2-...","{'chain_number_list': [1, 2, 3], 'chain_length_list': [83, 76], 'multimer_string_name': 'his-GG-1ubq-1-(<K48_1ubq-2-...","{'chain_number_list': [1, 2, 3, 4], 'chain_length_list': [83, 76, 76], 'multimer_string_name': 'his-GG-1ubq-1-(<K63_...","{'chain_number_list': [1, 2, 3, 4], 'chain_length_list': [83, 76, 76], 'multimer_string_name': 'his-GG-1ubq-1-(<K63_...","{'chain_number_list': [1, 2, 3, 4, 5], 'chain_length_list': [83, 76, 76, 76], 'multimer_string_name': 'his-GG-1ubq-1...","{'chain_number_list': [1, 2, 3, 4, 5], 'chain_length_list': [83, 76, 76, 76], 'multimer_string_name': 'his-GG-1ubq-1..."
2,2,75,Ub4_4,0,"{'chain_number_list': [1, 2], 'chain_length_list': [83], 'multimer_string_name': 'his-GG-1ubq-1-()', 'nomenclature_w...","{'chain_number_list': [1, 2, 3], 'chain_length_list': [83, 76], 'multimer_string_name': 'his-GG-1ubq-1-(<K48_1ubq-2-...","{'chain_number_list': [1, 2, 3], 'chain_length_list': [83, 76], 'multimer_string_name': 'his-GG-1ubq-1-(<K48_1ubq-2-...","{'chain_number_list': [1, 2, 3, 4], 'chain_length_list': [83, 76, 76], 'multimer_string_name': 'his-GG-1ubq-1-(<K63_...","{'chain_number_list': [1, 2, 3, 4], 'chain_length_list': [83, 76, 76], 'multimer_string_name': 'his-GG-1ubq-1-(<K63_...","{'chain_number_list': [1, 2, 3, 4, 5], 'chain_length_list': [83, 76, 76, 76], 'multimer_string_name': 'his-GG-1ubq-1...","{'chain_number_list': [1, 2, 3, 4, 5], 'chain_length_list': [83, 76, 76, 76], 'multimer_string_name': 'his-GG-1ubq-1..."



==> multimer_size_4/donor_history shape=(26, 10)


...

,Unnamed: 0,index,multimer_id,used_in_synthesis,initial_acceptor,dimer_formation,dimer_deprotection,trimer_formation,trimer_deprotection,tetramer_formation,tetramer_deprotection,pentamer_formation
0,0,47,Ub5_5,1,NaN,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...",NaN,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...",NaN,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...",NaN,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD..."
1,1,71,Ub5_5,0,NaN,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...",NaN,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...",NaN,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...",NaN,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD..."
2,2,91,Ub5_5,0,NaN,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...",NaN,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...",NaN,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...",NaN,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD..."



==> multimer_size_5/reaction_history shape=(114, 12)


,Unnamed: 0,index,multimer_id,used_in_synthesis,initial_acceptor,dimer_formation,dimer_deprotection,trimer_formation,trimer_deprotection,tetramer_formation,tetramer_deprotection,pentamer_formation
0,0,47,Ub5_5,1,NaN,gp78/Ube2g2,SMAC_deprot,gp78/Ube2g2,SMAC_deprot,gp78/Ube2g2,FAKE_deprot,Ubc13/Mms2 (branching)
1,1,71,Ub5_5,0,NaN,gp78/Ube2g2,SMAC_deprot,gp78/Ube2g2,SMAC_deprot,Ubc13/Mms2 (branching),FAKE_deprot,gp78/Ube2g2
2,2,91,Ub5_5,0,NaN,gp78/Ube2g2,SMAC_deprot,gp78/Ube2g2,FAKE_deprot,Ubc13/Mms2 (branching),SMAC_deprot,gp78/Ube2g2



==> multimer_size_5/ubiquitin_history shape=(114, 13)


,Unnamed: 0,index,multimer_id,used_in_synthesis,initial_acceptor,dimer_formation,dimer_deprotection,trimer_formation,trimer_deprotection,tetramer_formation,tetramer_deprotection,pentamer_formation,final_multimer
0,0,47,Ub5_5,1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD..."
1,1,71,Ub5_5,0,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD..."
2,2,91,Ub5_5,0,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD..."


# Named handles for frequently used tables

We expose convenient variables for common tables (if present).

In [443]:
# Convenience references (present only if keys exist)
ms4_combined = datasets.get("multimer_size_4/combined_database")
ms4_context  = datasets.get("multimer_size_4/context_history")
ms4_donor    = datasets.get("multimer_size_4/donor_history")
ms4_reaction = datasets.get("multimer_size_4/reaction_history")
ms4_ubq      = datasets.get("multimer_size_4/ubiquitin_history")

ms5_combined = datasets.get("multimer_size_5/combined_database")
ms5_context  = datasets.get("multimer_size_5/context_history")
ms5_donor    = datasets.get("multimer_size_5/donor_history")
ms5_reaction = datasets.get("multimer_size_5/reaction_history")
ms5_ubq      = datasets.get("multimer_size_5/ubiquitin_history")

for name, df in [
    ("ms4_combined", ms4_combined),
    ("ms5_combined", ms5_combined),
]:
    if df is not None:
        print(f"{name}: shape={df.shape}")

ms4_combined: shape=(78, 12)
ms5_combined: shape=(342, 14)


# Build Topology-Only Tables (No Protecting Groups)

We create topology-only versions of the tetramer and pentamer ubiquitin history tables by applying `ubiquitin_simulation` with `GLOBAL_deprot` to remove all protecting groups from the JSON structures. This produces clean topology representations that focus purely on the linkage patterns without chemical modifications.

- **Input**: `ms4_ubq` (tetramer) and `ms5_ubq` (pentamer) with protecting groups
- **Output**: `ms4_ubq_topology_only` and `ms5_ubq_topology_only` with stripped protecting groups
- **Preservation**: `multimer_id` is preserved as a string identifier in both tables

In [444]:
# Build topology-only tables for tetramer and pentamer ubiquitin history (column-wise, no protecting groups)

# Import ubiquitin_simulation from back_end/src/main.py
from pathlib import Path
import sys
import pandas as pd

# Ensure REPO_ROOT exists from earlier cell; fall back to cwd if missing
try:
    REPO_ROOT
except NameError:
    REPO_ROOT = Path.cwd()

SRC_DIR = REPO_ROOT / 'back_end' / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

try:
    from main import ubiquitin_simulation
except Exception as e:
    print("ERROR: Could not import 'ubiquitin_simulation' from back_end/src/main.py:", e)
    ubiquitin_simulation = None

def to_topology(val):
    """Apply GLOBAL_deprot to remove protecting groups from ubiquitin JSON."""
    if pd.isna(val):
        return None
    try:
        result = ubiquitin_simulation(parent_dictionary=val, ubi_molecule_to_add='', type_of_reaction='GLOBAL_deprot')
        # ubiquitin_simulation returns a tuple; take only the first value
        if isinstance(result, tuple):
            return result[0]
        return result
    except Exception:
        return None

# Process tetramer ubiquitin history (ms4_ubq)
if ubiquitin_simulation is None:
    print("Skipping topology-only table creation due to import error.")
elif 'ms4_ubq' not in globals() or ms4_ubq is None:
    print("ERROR: 'ms4_ubq' not available. Ensure datasets are loaded earlier.")
else:
    df4 = ms4_ubq.copy()
    topo4_df = pd.DataFrame(index=df4.index)

    # Preserve multimer_id as string
    if 'multimer_id' in df4.columns:
        topo4_df['multimer_id'] = df4['multimer_id'].astype(str)

    for c in df4.columns:
        if c == 'multimer_id':
            continue
        s = df4[c]
        if s.dtype == 'object':
            topo4_df[c] = s.apply(to_topology)
        else:
            topo4_df[c] = s

    ms4_ubq_topology_only = topo4_df
    print("Tetramer topology-only table created for all columns (preview):")
    display(ms4_ubq_topology_only.head(10))

# Process pentamer ubiquitin history (ms5_ubq)
if ubiquitin_simulation is None:
    print("Skipping pentamer topology-only table creation due to import error.")
elif 'ms5_ubq' not in globals() or ms5_ubq is None:
    print("ERROR: 'ms5_ubq' not available. Ensure datasets are loaded earlier.")
else:
    df5 = ms5_ubq.copy()
    topo5_df = pd.DataFrame(index=df5.index)

    # Preserve multimer_id as string
    if 'multimer_id' in df5.columns:
        topo5_df['multimer_id'] = df5['multimer_id'].astype(str)

    for c in df5.columns:
        if c == 'multimer_id':
            continue
        s = df5[c]
        if s.dtype == 'object':
            topo5_df[c] = s.apply(to_topology)
        else:
            topo5_df[c] = s

    ms5_ubq_topology_only = topo5_df
    print("\nPentamer topology-only table created for all columns (preview):")
    display(ms5_ubq_topology_only.head(10))

Tetramer topology-only table created for all columns (preview):


,multimer_id,Unnamed: 0,index,used_in_synthesis,initial_acceptor,dimer_formation,dimer_deprotection,trimer_formation,trimer_deprotection,tetramer_formation,final_multimer
0,Ub4_4,0,31,1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD..."
1,Ub4_4,1,55,0,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD..."
2,Ub4_4,2,75,0,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD..."
3,Ub4_7,3,95,1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD..."
4,Ub4_8,4,111,0,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKE


Pentamer topology-only table created for all columns (preview):


,multimer_id,Unnamed: 0,index,used_in_synthesis,initial_acceptor,dimer_formation,dimer_deprotection,trimer_formation,trimer_deprotection,tetramer_formation,tetramer_deprotection,pentamer_formation,final_multimer
0,Ub5_5,0,47,1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD..."
1,Ub5_5,1,71,0,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD..."
2,Ub5_5,2,91,0,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD..."
3,Ub5_9,3,111,1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTL

# Load provided band intensity Excel files

We now directly load the two provided Excel files:
- `back_end/revisions/bandstrength_tetramer2.xlsx`
- `back_end/revisions/chart_1-42.xlsx`

We will read all sheets from each file to preserve full context.

In [445]:
# Read all sheets from the two Excel files
band_file_1 = REPO_ROOT / "back_end" / "revisions" / "bandstrength_tetramer2.xlsx"
band_file_2 = REPO_ROOT / "back_end" / "revisions" / "chart_1-42.xlsx"

print("Band intensity files:")
print("-", band_file_1.relative_to(REPO_ROOT), "exists=", band_file_1.exists())
print("-", band_file_2.relative_to(REPO_ROOT), "exists=", band_file_2.exists())

# renamed: bandstrength_sheets -> tetramer_bandstrength
tetramer_bandstrength = None
pentamer_bandstrength = None

if band_file_1.exists():
    tetramer_bandstrength = pd.read_excel(band_file_1, sheet_name=None)
else:
    print("WARNING: bandstrength_tetramer2.xlsx not found.")

if band_file_2.exists():
    pentamer_bandstrength = pd.read_excel(band_file_2, sheet_name=None)
else:
    print("WARNING: chart_1-42.xlsx not found.")

# Report sheet names and shapes
if tetramer_bandstrength is not None:
    print("\nbandstrength_tetramer2.xlsx sheets:")
    for sname, sdf in tetramer_bandstrength.items():
        print(f"- {sname}: shape={sdf.shape}")

if pentamer_bandstrength is not None:
    print("\nchart_1-42.xlsx sheets:")
    for sname, sdf in pentamer_bandstrength.items():
        print(f"- {sname}: shape={sdf.shape}")

Band intensity files:
- back_end/revisions/bandstrength_tetramer2.xlsx exists= True
- back_end/revisions/chart_1-42.xlsx exists= True

bandstrength_tetramer2.xlsx sheets:
- Sheet1: shape=(14, 9)

chart_1-42.xlsx sheets:
- Sheet1: shape=(42, 12)


In [446]:
# Preview selected sheets (first rows)

if tetramer_bandstrength is not None:
    for sname, sdf in tetramer_bandstrength.items():
        print("\nBandstrength ->", sname, f"shape={sdf.shape}")
        display(sdf.head(5))

if pentamer_bandstrength is not None:
    for sname, sdf in pentamer_bandstrength.items():
        print("\nChart ->", sname, f"shape={sdf.shape}")
        display(sdf.head(5))


Bandstrength -> Sheet1 shape=(14, 9)


,Unnamed: 0,higher bands,tetramer,K48 trimer,branched trimer,63->48 trimer,48->63 trimer,K63 trimer,dimer
0,Ub4-1,0,461373,234861.0,NaN,NaN,NaN,NaN,9867
1,Ub4-2,0,1195422,137206.0,NaN,NaN,NaN,NaN,39401
2,Ub4-3,0,1096284,NaN,NaN,NaN,347045.0,NaN,1054
3,Ub4-4,26272,1084096,181952.0,NaN,NaN,NaN,NaN,256
4,Ub4-5,753095,2507050,NaN,NaN,NaN,680085.0,NaN,18270



Chart -> Sheet1 shape=(42, 12)


,Unnamed: 0,higher bands,pentamer,Ub4-8 (コ）,Ub4-5 (zig-zag,Ub4-9 (bottom branch,Ub4-7 ©,Ub4-6 (hetro,br trimer,hetero trimer,linear trimer,dimer
0,Ub5-1,NaN,310170,NaN,611835.0,NaN,NaN,NaN,163310.0,NaN,NaN,152530
1,Ub5-2,318773.0,816850,NaN,189999.0,NaN,601183.0,NaN,205995.0,NaN,NaN,224347
2,Ub5-3,NaN,1938911,NaN,578569.0,NaN,423576.0,NaN,449254.0,NaN,NaN,447552
3,Ub5-4,371926.0,875432,NaN,719610.0,NaN,NaN,NaN,NaN,213996.0,NaN,6358
4,Ub5-5,173569.0,585931,NaN,505672.0,NaN,NaN,NaN,113522.0,NaN,NaN,12493


# Data Cleaning Tetramer Band Intensities

## Band intensities: combine trimer columns

We create a single column per sheet that aggregates all columns containing the word "trimer" (case-insensitive). This avoids fragmentation across multiple trimer-related columns.

In [447]:
# Aggregate trimer-related columns into a single column per sheet (row-wise sum)
# Process both tetramer and pentamer bandstrength data

import re

# Process tetramer bandstrength
if tetramer_bandstrength is None:
    print("ERROR: bandstrength_tetramer2.xlsx not loaded earlier.")
else:
    print("Processing tetramer bandstrength...")
    tetramer_bandstrength_combined = {}
    for sname, sdf in tetramer_bandstrength.items():
        cols = list(sdf.columns)
        # Identify trimer columns by name pattern (case-insensitive substring match)
        trimer_cols = [c for c in cols if re.search(r"trimer", str(c), flags=re.IGNORECASE)]
        if not trimer_cols:
            print(f"Sheet '{sname}': no columns matched 'trimer' — leaving as-is.")
            tetramer_bandstrength_combined[sname] = sdf.copy()
            continue

        # Ensure numeric for summation; non-numeric coerces to NaN
        sdf2 = sdf.copy()
        for c in trimer_cols:
            sdf2[c] = pd.to_numeric(sdf2[c], errors='coerce')

        # Strict per-row sum across ONLY the trimer columns
        sdf2['trimer'] = sdf2[trimer_cols].sum(axis=1, skipna=True)

        # Optionally drop the original trimer columns to avoid duplication
        sdf2 = sdf2.drop(columns=trimer_cols)

        # Reorder columns: place 'trimer' between 'tetramer' and 'dimer' if they exist
        current_cols = list(sdf2.columns)
        if 'tetramer' in current_cols and 'dimer' in current_cols and 'trimer' in current_cols:
            # Remove 'trimer' temporarily and reinsert at desired position
            current_cols.remove('trimer')
            tet_idx = current_cols.index('tetramer')
            # Insert after 'tetramer' and before 'dimer'
            insert_idx = tet_idx + 1
            current_cols.insert(insert_idx, 'trimer')
            sdf2 = sdf2[current_cols]
        else:
            # If either 'tetramer' or 'dimer' missing, keep current order
            pass

        tetramer_bandstrength_combined[sname] = sdf2
        print(f"Tetramer sheet '{sname}': combined {len(trimer_cols)} trimer columns into 'trimer' and positioned between 'tetramer' and 'dimer' when available.")
        with pd.option_context('display.max_rows', 10, 'display.max_columns', None):
            display(sdf2.head(10))

    # Replace original reference with combined for downstream use
    tetramer_bandstrength = tetramer_bandstrength_combined

# Process pentamer bandstrength
if pentamer_bandstrength is None:
    print("\nWARNING: pentamer_bandstrength not loaded earlier.")
else:
    print("\nProcessing pentamer bandstrength...")
    pentamer_bandstrength_combined = {}
    for sname, sdf in pentamer_bandstrength.items():
        sdf2 = sdf.copy()
        cols = list(sdf2.columns)
        
        # Identify Ub4-* columns (tetramer bands in pentamer file)
        tetramer_cols = [c for c in cols if c.startswith('Ub4-')]
        
        # Identify trimer columns by name pattern (case-insensitive substring match)
        trimer_cols = [c for c in cols if re.search(r"trimer", str(c), flags=re.IGNORECASE)]
        
        print(f"Sheet '{sname}': found {len(tetramer_cols)} tetramer columns (Ub4-*) and {len(trimer_cols)} trimer columns")
        
        # Process tetramer columns
        if tetramer_cols:
            # Ensure numeric for summation; non-numeric coerces to NaN
            for c in tetramer_cols:
                sdf2[c] = pd.to_numeric(sdf2[c], errors='coerce')
            # Sum tetramer columns
            sdf2['tetramer'] = sdf2[tetramer_cols].sum(axis=1, skipna=True)
            # Drop the original tetramer columns
            sdf2 = sdf2.drop(columns=tetramer_cols)
        
        # Process trimer columns
        if trimer_cols:
            # Ensure numeric for summation
            for c in trimer_cols:
                sdf2[c] = pd.to_numeric(sdf2[c], errors='coerce')
            # Sum trimer columns
            sdf2['trimer'] = sdf2[trimer_cols].sum(axis=1, skipna=True)
            # Drop the original trimer columns
            sdf2 = sdf2.drop(columns=trimer_cols)
        
        # Reorder columns: desired order is higher bands, pentamer, tetramer, trimer, dimer
        current_cols = list(sdf2.columns)
        desired_order = []
        
        # Build desired order if columns exist
        if 'Unnamed: 0' in current_cols:
            desired_order.append('Unnamed: 0')
        if 'higher bands' in current_cols:
            desired_order.append('higher bands')
        if 'pentamer' in current_cols:
            desired_order.append('pentamer')
        if 'tetramer' in current_cols:
            desired_order.append('tetramer')
        if 'trimer' in current_cols:
            desired_order.append('trimer')
        if 'dimer' in current_cols:
            desired_order.append('dimer')
        
        # Add any remaining columns not in desired order
        remaining = [c for c in current_cols if c not in desired_order]
        final_cols = desired_order + remaining
        sdf2 = sdf2[final_cols]

        pentamer_bandstrength_combined[sname] = sdf2
        print(f"Pentamer sheet '{sname}': combined {len(tetramer_cols)} Ub4-* columns into 'tetramer' and {len(trimer_cols)} trimer columns into 'trimer'")
        with pd.option_context('display.max_rows', 10, 'display.max_columns', None):
            display(sdf2.head(10))

    # Replace original reference with combined for downstream use
    pentamer_bandstrength = pentamer_bandstrength_combined

Processing tetramer bandstrength...
Tetramer sheet 'Sheet1': combined 5 trimer columns into 'trimer' and positioned between 'tetramer' and 'dimer' when available.


,Unnamed: 0,higher bands,tetramer,trimer,dimer
0,Ub4-1,0,461373,234861.0,9867
1,Ub4-2,0,1195422,137206.0,39401
2,Ub4-3,0,1096284,347045.0,1054
3,Ub4-4,26272,1084096,181952.0,256
4,Ub4-5,753095,2507050,680085.0,18270
5,Ub4-6,975098,2362117,1564249.0,2109
6,Ub4-7,267614,595272,15538.0,918
7,Ub4-8,1171975,2280495,229845.0,735
8,Ub4-9,2210574,2721864,339568.0,3002
9,Ub4-10,465552,1456488,339048.0,68904



Processing pentamer bandstrength...
Sheet 'Sheet1': found 5 tetramer columns (Ub4-*) and 3 trimer columns
Pentamer sheet 'Sheet1': combined 5 Ub4-* columns into 'tetramer' and 3 trimer columns into 'trimer'


,Unnamed: 0,higher bands,pentamer,tetramer,trimer,dimer
0,Ub5-1,NaN,310170,611835.0,163310.0,152530
1,Ub5-2,318773.0,816850,791182.0,205995.0,224347
2,Ub5-3,NaN,1938911,1002145.0,449254.0,447552
3,Ub5-4,371926.0,875432,719610.0,213996.0,6358
4,Ub5-5,173569.0,585931,505672.0,113522.0,12493
5,Ub5-6,NaN,1229576,449616.0,418574.0,100300
6,Ub5-7,25200.0,1047305,431165.0,9380.0,262185
7,Ub5-8,172340.0,1813560,640815.0,576170.0,2030
8,Ub5-9,923010.0,1056627,278685.0,0.0,1221
9,Ub5-10,1335928.0,2294796,479196.0,21420.0,986


# Band intensities: compute yield

We add a `yield` column per sheet defined as:

yield = tetramer / (higher bands + tetramer + trimer + dimer)

We use the existing column named `higher bands` directly; missing values are treated as 0 and division-by-zero is avoided.

In [448]:
# Calculate yield and overreaction per sheet for both tetramer and pentamer bandstrength
import re

# Process tetramer bandstrength
if tetramer_bandstrength is None:
    print("ERROR: tetramer_bandstrength not available (load step may not have run).")
else:
    print("Computing yield/overreaction for tetramer bandstrength...")
    tetramer_bandstrength_with_yield = {}
    for sname, sdf in tetramer_bandstrength.items():
        df = sdf.copy()
        # Ensure numeric for involved columns
        for col in ['tetramer', 'trimer', 'dimer', 'higher bands']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            else:
                # create missing columns defaulting to 0
                df[col] = 0.0

        denom = df['higher bands'].fillna(0) + df['tetramer'].fillna(0) + df['trimer'].fillna(0) + df['dimer'].fillna(0)
        df['yield'] = np.where(denom > 0, df['tetramer'].fillna(0) / denom, np.nan)

        # overreaction = higher bands / (higher bands + tetramer + trimer + dimer)
        df['overreaction'] = np.where(denom > 0, df['higher bands'].fillna(0) / denom, np.nan)

        # Reorder columns to desired final order, retaining any other columns after
        desired_prefix = ['Unnamed: 0', 'higher bands', 'tetramer', 'trimer', 'dimer', 'yield', 'overreaction']
        # Some files may use 'Unnamed:0' without space; normalize detection
        has_unnamed_space = 'Unnamed: 0' in df.columns
        has_unnamed_nospace = 'Unnamed:0' in df.columns
        if has_unnamed_nospace and not has_unnamed_space:
            # Temporarily align to expected label for ordering preview; keep original name
            desired_order = ['Unnamed:0', 'higher bands', 'tetramer', 'trimer', 'dimer', 'yield', 'overreaction']
        else:
            desired_order = desired_prefix
        # Build final column order list
        existing_desired = [c for c in desired_order if c in df.columns]
        remaining = [c for c in df.columns if c not in existing_desired]
        final_cols = existing_desired + remaining
        df = df[final_cols]

        tetramer_bandstrength_with_yield[sname] = df
        print(f"Tetramer sheet '{sname}': yield/overreaction computed and columns ordered as {existing_desired} (+ remaining).")
        display(df.head(10))

    # Update reference for downstream use
    tetramer_bandstrength = tetramer_bandstrength_with_yield

# Process pentamer bandstrength
if pentamer_bandstrength is None:
    print("\nWARNING: pentamer_bandstrength not available.")
else:
    print("\nComputing yield/overreaction for pentamer bandstrength...")
    pentamer_bandstrength_with_yield = {}
    for sname, sdf in pentamer_bandstrength.items():
        df = sdf.copy()
        # Ensure numeric for involved columns (pentamer includes pentamer + tetramer)
        for col in ['pentamer', 'tetramer', 'trimer', 'dimer', 'higher bands']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            else:
                # create missing columns defaulting to 0
                df[col] = 0.0
        
        # Convert NaN to 0 in all band columns for pentamers
        for col in ['pentamer', 'tetramer', 'trimer', 'dimer', 'higher bands']:
            if col in df.columns:
                df[col] = df[col].fillna(0)

        # Denominator includes all band types
        denom = df['higher bands'] + df['pentamer'] + df['tetramer'] + df['trimer'] + df['dimer']
        
        # Yield = pentamer / (all bands)
        df['yield'] = np.where(denom > 0, df['pentamer'] / denom, np.nan)

        # Overreaction = higher bands / (all bands)
        df['overreaction'] = np.where(denom > 0, df['higher bands'] / denom, np.nan)

        # Reorder columns to desired final order
        desired_prefix = ['Unnamed: 0', 'higher bands', 'pentamer', 'tetramer', 'trimer', 'dimer', 'yield', 'overreaction']
        # Some files may use 'Unnamed:0' without space; normalize detection
        has_unnamed_space = 'Unnamed: 0' in df.columns
        has_unnamed_nospace = 'Unnamed:0' in df.columns
        if has_unnamed_nospace and not has_unnamed_space:
            desired_order = ['Unnamed:0', 'higher bands', 'pentamer', 'tetramer', 'trimer', 'dimer', 'yield', 'overreaction']
        else:
            desired_order = desired_prefix
        # Build final column order list
        existing_desired = [c for c in desired_order if c in df.columns]
        remaining = [c for c in df.columns if c not in existing_desired]
        final_cols = existing_desired + remaining
        df = df[final_cols]

        pentamer_bandstrength_with_yield[sname] = df
        print(f"Pentamer sheet '{sname}': yield/overreaction computed and columns ordered as {existing_desired} (+ remaining).")
        display(df.head(10))

    # Update reference for downstream use
    pentamer_bandstrength = pentamer_bandstrength_with_yield

Computing yield/overreaction for tetramer bandstrength...
Tetramer sheet 'Sheet1': yield/overreaction computed and columns ordered as ['Unnamed: 0', 'higher bands', 'tetramer', 'trimer', 'dimer', 'yield', 'overreaction'] (+ remaining).


,Unnamed: 0,higher bands,tetramer,trimer,dimer,yield,overreaction
0,Ub4-1,0,461373,234861.0,9867,0.653409,0.000000
1,Ub4-2,0,1195422,137206.0,39401,0.871280,0.000000
2,Ub4-3,0,1096284,347045.0,1054,0.758998,0.000000
3,Ub4-4,26272,1084096,181952.0,256,0.838710,0.020325
4,Ub4-5,753095,2507050,680085.0,18270,0.633333,0.190248
5,Ub4-6,975098,2362117,1564249.0,2109,0.481713,0.198855
6,Ub4-7,267614,595272,15538.0,918,0.676952,0.304334
7,Ub4-8,1171975,2280495,229845.0,735,0.619187,0.318208
8,Ub4-9,2210574,2721864,339568.0,3002,0.515992,0.419066
9,Ub4-10,465552,1456488,339048.0,68904,0.625104,0.199808



Computing yield/overreaction for pentamer bandstrength...
Pentamer sheet 'Sheet1': yield/overreaction computed and columns ordered as ['Unnamed: 0', 'higher bands', 'pentamer', 'tetramer', 'trimer', 'dimer', 'yield', 'overreaction'] (+ remaining).


,Unnamed: 0,higher bands,pentamer,tetramer,trimer,dimer,yield,overreaction
0,Ub5-1,0.0,310170,611835.0,163310.0,152530,0.250573,0.000000
1,Ub5-2,318773.0,816850,791182.0,205995.0,224347,0.346542,0.135237
2,Ub5-3,0.0,1938911,1002145.0,449254.0,447552,0.505206,0.000000
3,Ub5-4,371926.0,875432,719610.0,213996.0,6358,0.400230,0.170037
4,Ub5-5,173569.0,585931,505672.0,113522.0,12493,0.421173,0.124763
5,Ub5-6,0.0,1229576,449616.0,418574.0,100300,0.559390,0.000000
6,Ub5-7,25200.0,1047305,431165.0,9380.0,262185,0.589953,0.014195
7,Ub5-8,172340.0,1813560,640815.0,576170.0,2030,0.565868,0.053774
8,Ub5-9,923010.0,1056627,278685.0,0.0,1221,0.467629,0.408494
9,Ub5-10,1335928.0,2294796,479196.0,21420.0,986,0.555328,0.323287


In [449]:
# Normalize identifier column name for tetramer bandstrength sheets
# Rename 'Unnamed: 0' or 'Unnamed:0' to 'multimer_id' in all sheets
# Also normalize multimer_id values: replace '-' with '_' (e.g., UbX-Y -> UbX_Y)

if tetramer_bandstrength is None:
    print("ERROR: tetramer_bandstrength dictionary not available.")
else:
    renamed_sheets = {}
    for sname, df in tetramer_bandstrength.items():
        cur = df.copy()
        # Detect either variant and rename to 'multimer_id'
        if 'Unnamed: 0' in cur.columns:
            cur = cur.rename(columns={'Unnamed: 0': 'multimer_id'})
        elif 'Unnamed:0' in cur.columns:
            cur = cur.rename(columns={'Unnamed:0': 'multimer_id'})
        
        # Normalize multimer_id values: replace '-' with '_'
        if 'multimer_id' in cur.columns:
            cur['multimer_id'] = cur['multimer_id'].astype(str).str.replace('-', '_', regex=False)
            # Ensure 'multimer_id' is first
            cols = list(cur.columns)
            cols.remove('multimer_id')
            cur = cur[['multimer_id'] + cols]
        
        renamed_sheets[sname] = cur
        print(f"Sheet '{sname}': identifier column normalized to 'multimer_id' and values converted from '-' to '_'.")
        display(cur.head(10))
    tetramer_bandstrength = renamed_sheets

# Normalize identifier column name for pentamer bandstrength sheets
# Rename 'Unnamed: 0' or 'Unnamed:0' to 'multimer_id' in all sheets
# Also normalize multimer_id values: replace '-' with '_' (e.g., UbX-Y -> UbX_Y)

if pentamer_bandstrength is None:
    print("\nWARNING: pentamer_bandstrength dictionary not available.")
else:
    renamed_sheets_pentamer = {}
    for sname, df in pentamer_bandstrength.items():
        cur = df.copy()
        # Detect either variant and rename to 'multimer_id'
        if 'Unnamed: 0' in cur.columns:
            cur = cur.rename(columns={'Unnamed: 0': 'multimer_id'})
        elif 'Unnamed:0' in cur.columns:
            cur = cur.rename(columns={'Unnamed:0': 'multimer_id'})
        
        # Normalize multimer_id values: replace '-' with '_'
        if 'multimer_id' in cur.columns:
            cur['multimer_id'] = cur['multimer_id'].astype(str).str.replace('-', '_', regex=False)
            # Ensure 'multimer_id' is first
            cols = list(cur.columns)
            cols.remove('multimer_id')
            cur = cur[['multimer_id'] + cols]
        
        renamed_sheets_pentamer[sname] = cur
        print(f"Pentamer sheet '{sname}': identifier column normalized to 'multimer_id' and values converted from '-' to '_'.")
        display(cur.head(10))
    pentamer_bandstrength = renamed_sheets_pentamer

Sheet 'Sheet1': identifier column normalized to 'multimer_id' and values converted from '-' to '_'.


,multimer_id,higher bands,tetramer,trimer,dimer,yield,overreaction
0,Ub4_1,0,461373,234861.0,9867,0.653409,0.000000
1,Ub4_2,0,1195422,137206.0,39401,0.871280,0.000000
2,Ub4_3,0,1096284,347045.0,1054,0.758998,0.000000
3,Ub4_4,26272,1084096,181952.0,256,0.838710,0.020325
4,Ub4_5,753095,2507050,680085.0,18270,0.633333,0.190248
5,Ub4_6,975098,2362117,1564249.0,2109,0.481713,0.198855
6,Ub4_7,267614,595272,15538.0,918,0.676952,0.304334
7,Ub4_8,1171975,2280495,229845.0,735,0.619187,0.318208
8,Ub4_9,2210574,2721864,339568.0,3002,0.515992,0.419066
9,Ub4_10,465552,1456488,339048.0,68904,0.625104,0.199808


Pentamer sheet 'Sheet1': identifier column normalized to 'multimer_id' and values converted from '-' to '_'.


,multimer_id,higher bands,pentamer,tetramer,trimer,dimer,yield,overreaction
0,Ub5_1,0.0,310170,611835.0,163310.0,152530,0.250573,0.000000
1,Ub5_2,318773.0,816850,791182.0,205995.0,224347,0.346542,0.135237
2,Ub5_3,0.0,1938911,1002145.0,449254.0,447552,0.505206,0.000000
3,Ub5_4,371926.0,875432,719610.0,213996.0,6358,0.400230,0.170037
4,Ub5_5,173569.0,585931,505672.0,113522.0,12493,0.421173,0.124763
5,Ub5_6,0.0,1229576,449616.0,418574.0,100300,0.559390,0.000000
6,Ub5_7,25200.0,1047305,431165.0,9380.0,262185,0.589953,0.014195
7,Ub5_8,172340.0,1813560,640815.0,576170.0,2030,0.565868,0.053774
8,Ub5_9,923010.0,1056627,278685.0,0.0,1221,0.467629,0.408494
9,Ub5_10,1335928.0,2294796,479196.0,21420.0,986,0.555328,0.323287


# Summary: Available Datasets for Analysis

We now have the following key datasets prepared for our machine learning analysis:

## Band Intensity Data
- **`tetramer_bandstrength`**: Normalized tetramer band intensity measurements with computed yield and overreaction metrics
- **`pentamer_bandstrength`**: TODO - pentamer band intensity data (to be added)

## Ubiquitin History Tables (With Protecting Groups)
- **`ms4_ubq`**: Tetramer ubiquitin reaction history including protecting group information
- **`ms5_ubq`**: Pentamer ubiquitin reaction history including protecting group information

## Topology-Only Tables (No Protecting Groups)
- **`ms4_ubq_topology_only`**: Clean tetramer topology without chemical modifications
- **`ms5_ubq_topology_only`**: Clean pentamer topology without chemical modifications

---

**Analysis Goal**: Use these tables to build a machine learning pipeline that identifies the most important ubiquitin topologies (linkage patterns) affecting reaction yield. By removing protecting groups in the topology-only tables, we can focus purely on structural features that drive synthesis outcomes.

# Analysis Configuration

Configure which dataset to analyze by setting the parameters below:
- **Multimer size**: Choose between tetramers (4) or pentamers (5)
- **Topology mode**: Choose whether to use topology-only tables (no protecting groups) or full tables (with protecting groups)

In [450]:
# Configuration: Select multimer size and topology mode

# ============================================
# USER CONFIGURATION
# ============================================

# Choose multimer size: 4 (tetramer) or 5 (pentamer)
MULTIMER_SIZE = 5

# Choose topology mode: True (topology-only, no protecting groups) or False (with protecting groups)
TOPOLOGY_ONLY = True

# Choose target variable for Random Forest: 'yield' or 'overreaction'
TARGET_VARIABLE = 'yield'

# ============================================
# AUTO-CONFIGURATION (based on above choices)
# ============================================

if MULTIMER_SIZE == 4:
    # Tetramer configuration
    bandstrength_data = tetramer_bandstrength if 'tetramer_bandstrength' in globals() else None
    if TOPOLOGY_ONLY:
        ubq_data = ms4_ubq_topology_only if 'ms4_ubq_topology_only' in globals() else None
        analysis_label = "Tetramer (Topology-Only)"
    else:
        ubq_data = ms4_ubq if 'ms4_ubq' in globals() else None
        analysis_label = "Tetramer (With Protecting Groups)"
elif MULTIMER_SIZE == 5:
    # Pentamer configuration
    bandstrength_data = None  # TODO: pentamer_bandstrength_merged not yet created
    if TOPOLOGY_ONLY:
        ubq_data = ms5_ubq_topology_only if 'ms5_ubq_topology_only' in globals() else None
        analysis_label = "Pentamer (Topology-Only)"
    else:
        ubq_data = ms5_ubq if 'ms5_ubq' in globals() else None
        analysis_label = "Pentamer (With Protecting Groups)"
else:
    raise ValueError(f"Invalid MULTIMER_SIZE={MULTIMER_SIZE}. Must be 4 or 5.")

# Filter to only rows where used_in_synthesis == 1
if ubq_data is not None:
    if 'used_in_synthesis' in ubq_data.columns:
        ubq_data = ubq_data[ubq_data['used_in_synthesis'] == 1].copy()
        print(f"Filtered to rows with used_in_synthesis == 1")
    else:
        print(f"WARNING: 'used_in_synthesis' column not found in ubq_data")

# Validation
print(f"\n{'='*60}")
print(f"ANALYSIS CONFIGURATION: {analysis_label}")
print(f"{'='*60}")
print(f"Multimer size:        {MULTIMER_SIZE}")
print(f"Topology-only mode:   {TOPOLOGY_ONLY}")
print(f"Ubiquitin data:       {'Available' if ubq_data is not None else 'NOT AVAILABLE'}")
print(f"Bandstrength data:    {'Available' if bandstrength_data is not None else 'NOT AVAILABLE'}")

if ubq_data is not None:
    print(f"Ubiquitin data shape: {ubq_data.shape}")
    display(ubq_data.head(5))
else:
    print("\n⚠️  WARNING: Ubiquitin data not available for this configuration.")

if bandstrength_data is not None:
    print(f"\nBandstrength sheets:  {list(bandstrength_data.keys())}")
else:
    print("\n⚠️  WARNING: Bandstrength data not available for this configuration.")
    if MULTIMER_SIZE == 5:
        print("   → Pentamer bandstrength processing is marked as TODO.")
print(f"{'='*60}\n")


Filtered to rows with used_in_synthesis == 1

ANALYSIS CONFIGURATION: Pentamer (Topology-Only)
Multimer size:        5
Topology-only mode:   True
Ubiquitin data:       Available
Bandstrength data:    NOT AVAILABLE
Ubiquitin data shape: (42, 13)


,multimer_id,Unnamed: 0,index,used_in_synthesis,initial_acceptor,dimer_formation,dimer_deprotection,trimer_formation,trimer_deprotection,tetramer_formation,tetramer_deprotection,pentamer_formation,final_multimer
0,Ub5_5,0,47,1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD..."
3,Ub5_9,3,111,1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD..."
4,Ub5_13,4,127,1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD..."
17,Ub5_22,17,411,1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQ


⚠️  WARNING: Bandstrength data not available for this configuration.
   → Pentamer bandstrength processing is marked as TODO.



# Data Preparation: Merge Ubiquitin Features with Band Intensity

Now that we've configured our analysis parameters, we prepare the data for machine learning through a 5-step pipeline. Each step is documented in detail below with its own markdown cell and code implementation.

The pipeline transforms raw ubiquitin history data and band intensity measurements into a unified dataset suitable for Random Forest analysis:

1. **Clean Ubiquitin Data** → Remove metadata, standardize formats
2. **Collect Unique Values** → Build vocabulary of all unique structures
3. **Encode Features** → Map structures to numeric codes
4. **Create Feature Count Matrix** → Count occurrences per multimer
5. **Merge with Band Intensity** → Combine features with target variables

**Final Output**: `bandstrength_data_merged` - a dataset containing target variables (`yield`, `overreaction`) and feature variables (encoded ubiquitin topology counts) ready for machine learning.

## Step 1: Clean Ubiquitin Data

Remove metadata columns and convert all columns to string type for consistent encoding.
- **Input**: `ubq_data` (from configuration)
- **Output**: `ubq_data_clean` - cleaned ubiquitin history table

In [451]:
# Step 1: Clean ubiquitin data - remove metadata columns

if ubq_data is None:
    print("ERROR: ubq_data not available. Run the configuration cell first.")
else:
    clean_ubq = ubq_data.copy()
    cols_to_drop = [c for c in ['Unnamed: 0', 'Unnamed:0', 'index', 'used_in_synthesis', 'initial_acceptor'] if c in clean_ubq.columns]
    if cols_to_drop:
        clean_ubq = clean_ubq.drop(columns=cols_to_drop)
        print(f"Dropped columns: {cols_to_drop}")
    else:
        print("No metadata columns found to drop.")

    # Convert all columns to string type (keep multimer_id as string, convert all others)
    for col in clean_ubq.columns:
        clean_ubq[col] = clean_ubq[col].astype(str)
    
    print(f"Converted all columns to string type.")
    print(f"\nCleaned ubiquitin data preview ({analysis_label}):")
    display(clean_ubq.head(10))

    # Store for downstream use
    ubq_data_clean = clean_ubq


Dropped columns: ['Unnamed: 0', 'index', 'used_in_synthesis', 'initial_acceptor']
Converted all columns to string type.

Cleaned ubiquitin data preview (Pentamer (Topology-Only)):


,multimer_id,dimer_formation,dimer_deprotection,trimer_formation,trimer_deprotection,tetramer_formation,tetramer_deprotection,pentamer_formation,final_multimer
0,Ub5_5,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD..."
3,Ub5_9,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD..."
4,Ub5_13,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD..."
17,Ub5_22,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","{'protein': '1ubq', 'c

## Step 2: Collect Unique Values

Scan all non-ID columns and collect unique JSON/string values across the entire table.
- **Input**: `ubq_data_clean`
- **Output**: `ubq_values_excluding_id` - sorted list of unique values

In [452]:
# Step 2: Collect unique values for encoding

if 'ubq_data_clean' not in globals() or ubq_data_clean is None:
    print("ERROR: ubq_data_clean is not available. Run the cleaning cell first.")
else:
    cur = ubq_data_clean.copy()
    exclude_col = 'multimer_id' if 'multimer_id' in cur.columns else None
    cols_to_scan = [c for c in cur.columns if c != exclude_col]

    unique_vals = set()
    for c in cols_to_scan:
        try:
            vals = cur[c].dropna().unique().tolist()
        except Exception:
            vals = []
        for v in vals:
            unique_vals.add(v)

    ubq_values_excluding_id = sorted(list(unique_vals))
    print(f"Collected {len(ubq_values_excluding_id)} unique values from ubq_data_clean excluding '{exclude_col or 'N/A'}'.")
    print(f"Configuration: {analysis_label}")
    # Preview first 5 values for readability (they will be long strings)
    preview = ubq_values_excluding_id[:5]
    print(f"Preview (first 5 of {len(ubq_values_excluding_id)}):")
    for i, val in enumerate(preview, 1):
        print(f"{i}. {val[:150]}..." if len(val) > 150 else f"{i}. {val}")


Collected 63 unique values from ubq_data_clean excluding 'multimer_id'.
Configuration: Pentamer (Topology-Only)
Preview (first 5 of 63):
1. {'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGGDHHHHHH', 'chain...
2. {'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGGDHHHHHH', 'chain...
3. {'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGGDHHHHHH', 'chain...
4. {'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGGDHHHHHH', 'chain...
5. {'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGGDHHHHHH', 'chain...


## Step 3: Encode Features

Assign numeric codes to each unique value and apply the mapping to convert all cells to numeric codes.
- **Input**: `ubq_values_excluding_id`, `ubq_data_clean`
- **Output**: `ubq_data_encoded` - encoded table, `ubq_value_to_code_map` - code mapping dictionary

In [453]:
# Step 3: Assign numeric codes to each unique value (excluding 'multimer_id')

if 'ubq_values_excluding_id' not in globals() or ubq_values_excluding_id is None:
    print("ERROR: ubq_values_excluding_id not available. Run the previous collection cell.")
elif 'ubq_data_clean' not in globals() or ubq_data_clean is None:
    print("ERROR: ubq_data_clean not available. Run the cleaning cell first.")
else:
    # Build mapping dict: value -> integer code starting at 1
    ubq_value_to_code = {val: idx for idx, val in enumerate(ubq_values_excluding_id, start=1)}
    print(f"Created value-to-code mapping for {len(ubq_value_to_code)} values ({analysis_label})")
    print(f"Example slice (first 5 mappings):")
    example_items = list(ubq_value_to_code.items())[:5]
    for val, code in example_items:
        val_preview = val[:100] + "..." if len(val) > 100 else val
        print(f"  {code}: {val_preview}")

    # Apply mapping to a copy of the cleaned dataframe (excluding 'multimer_id')
    df = ubq_data_clean.copy()
    exclude_col = 'multimer_id' if 'multimer_id' in df.columns else None
    cols_to_encode = [c for c in df.columns if c != exclude_col]

    encoded_df = df.copy()
    for c in cols_to_encode:
        # Map string values to numeric codes
        encoded_df[c] = encoded_df[c].map(ubq_value_to_code)

    # Keep identifier first if present
    if exclude_col:
        ordered_cols = [exclude_col] + [c for c in encoded_df.columns if c != exclude_col]
        encoded_df = encoded_df[ordered_cols]

    # Store results for downstream use
    ubq_value_to_code_map = ubq_value_to_code
    ubq_data_encoded = encoded_df

    print(f"\nEncoded DataFrame preview ({analysis_label}):")
    display(ubq_data_encoded.head(10))


Created value-to-code mapping for 63 values (Pentamer (Topology-Only))
Example slice (first 5 mappings):
  1: {'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQR...
  2: {'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQR...
  3: {'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQR...
  4: {'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQR...
  5: {'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQR...

Encoded DataFrame preview (Pentamer (Topology-Only)):


,multimer_id,dimer_formation,dimer_deprotection,trimer_formation,trimer_deprotection,tetramer_formation,tetramer_deprotection,pentamer_formation,final_multimer
0,Ub5_5,1,1,2,2,3,3,26,26
3,Ub5_9,1,1,2,2,25,25,27,27
4,Ub5_13,1,1,2,2,25,25,34,34
17,Ub5_22,1,1,24,24,28,28,35,35
19,Ub5_21,1,1,24,24,28,28,31,31
21,Ub5_26,1,1,24,24,46,46,50,50
29,Ub5_24,23,23,24,24,33,33,37,37
30,Ub5_25,23,23,24,24,33,33,41,41
37,Ub5_27,23,23,45,45,46,46,54,54
39,Ub5_28,23,23,45,45,58,58,59,59


## Step 4: Create Feature Count Matrix

For each unique code, count how many times it appears per row (multimer) to create a feature count matrix.
- **Input**: `ubq_data_encoded`
- **Output**: `ubq_data_value_counts` - feature count matrix with columns '1', '2', '3', ... and `multimer_id`

In [454]:
# Step 4: Expand encoded values into columns; count occurrences per row (excluding 'multimer_id')

if 'ubq_data_encoded' not in globals() or ubq_data_encoded is None:
    print("ERROR: ubq_data_encoded is not available. Run the encoding cell first.")
else:
    cur_enc = ubq_data_encoded.copy()
    id_col = 'multimer_id' if 'multimer_id' in cur_enc.columns else None
    cols_to_scan = [c for c in cur_enc.columns if c != id_col]

    # Collect unique numeric codes from non-ID columns
    unique_codes = set()
    for c in cols_to_scan:
        vals = pd.to_numeric(cur_enc[c], errors='coerce')
        unique_codes.update([v for v in vals.dropna().unique().tolist()])
    # Ensure integer headers, sorted
    unique_codes = sorted(int(v) for v in unique_codes)
    print(f"Found {len(unique_codes)} unique encoded values across non-ID columns ({analysis_label})")

    # Initialize the output with the identifier column first (if present)
    out_df = pd.DataFrame()
    if id_col:
        out_df[id_col] = cur_enc[id_col]

    # For each encoded value, create a column counting occurrences across row's non-ID columns
    for code in unique_codes:
        out_df[str(code)] = (cur_enc[cols_to_scan] == code).sum(axis=1)

    # Store for downstream use and preview
    ubq_data_value_counts = out_df
    print(f"\nEncoded value-counts matrix preview ({analysis_label}):")
    display(ubq_data_value_counts.head(10))


Found 63 unique encoded values across non-ID columns (Pentamer (Topology-Only))

Encoded value-counts matrix preview (Pentamer (Topology-Only)):


,multimer_id,1,2,3,4,5,6,7,8,9,...,54,55,56,57,58,59,60,61,62,63
0,Ub5_5,2,2,2,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Ub5_9,2,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Ub5_13,2,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
17,Ub5_22,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
19,Ub5_21,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
21,Ub5_26,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
29,Ub5_24,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
30,Ub5_25,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
37,Ub5_27,0,0,0,0,0,0,0,0,0,...,2,0,0,0,0,0,0,0,0,0
39,Ub5_28,0,0,0,0,0,0,0,0,0,...,0,0,0,0,2,2,0,0,0,0


## Step 5: Merge Ubiquitin Value-Counts with Band Intensity Data

This step merges the per-row value-counts matrix (`ubq_data_value_counts`) into each bandstrength sheet, keyed by `multimer_id`.
- ID normalization: bandstrength sheet IDs use `-` while `ubq_data_value_counts` uses `_`. We normalize bandstrength IDs by replacing `-` with `_` before merging.
- Join type: left merge on `multimer_id`, preserving all rows from the bandstrength sheet and appending the value-count columns.
- Output: `bandstrength_data_merged` - a dictionary with the merged DataFrame for each sheet, containing both band intensity measurements and encoded ubiquitin features.

In [455]:
if 'ubq_data_value_counts' not in globals() or ubq_data_value_counts is None:
    print("ERROR: ubq_data_value_counts not available. Run the value-counts cell first.")
elif 'bandstrength_data' not in globals() or bandstrength_data is None:
    print("ERROR: bandstrength_data not configured. Set MULTIMER_SIZE and run configuration cell.")
elif not isinstance(bandstrength_data, dict):
    print("ERROR: bandstrength_data must be a dictionary of DataFrames (one per sheet).")
else:
    merged_sheets = {}
    counts_df = ubq_data_value_counts.copy()
    if 'multimer_id' not in counts_df.columns:
        print("ERROR: 'multimer_id' missing in ubq_data_value_counts; cannot merge.")
    else:
        # Ensure multimer_id is string for safe replace
        counts_df['multimer_id'] = counts_df['multimer_id'].astype(str)
        print(f"Merging {len(counts_df)} rows of value-counts into bandstrength data ({analysis_label})...\n")
        
        for sname, sdf in bandstrength_data.items():
            df = sdf.copy()
            if 'multimer_id' not in df.columns:
                print(f"WARNING: Sheet '{sname}' missing 'multimer_id'; skipping merge.")
                merged_sheets[sname] = df
                continue
            # Normalize IDs in bandstrength sheet: '-' -> '_'
            df['multimer_id'] = df['multimer_id'].astype(str).str.replace('-', '_', regex=False)
            # Perform left merge to add counts columns
            merged = df.merge(counts_df, on='multimer_id', how='left', suffixes=('', '_ubqcounts'))
            merged_sheets[sname] = merged
            print(f"Sheet '{sname}': merged value-counts on normalized 'multimer_id'. Result shape={merged.shape}")
            display(merged.head(10))
        
        bandstrength_data_merged = merged_sheets
        print(f"\nMerge complete ({analysis_label}): {len(bandstrength_data_merged)} sheets merged.")    

ERROR: bandstrength_data not configured. Set MULTIMER_SIZE and run configuration cell.


# Tetramer Reaction Pathway Analysis

## Random Forest: Overreaction Feature Importance (Tetramers)

We train a Random Forest regressor per tetramer bandstrength sheet using `overreaction` as the target (`y`) and encoded value-count features (`'1'` through `'73'`) as predictors (`X`). Results include model fit info and sorted feature importances for interpretability.

In [456]:
# Train Random Forest regressors to quantify feature importance for overreaction

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
import pandas as pd

# Expect bandstrength_data_merged to be a dict of DataFrames per sheet
if 'bandstrength_data_merged' not in globals() or bandstrength_data_merged is None:
    print("ERROR: bandstrength_data_merged not available. Run the merge step first.")
elif 'ubq_data_value_counts' not in globals() or ubq_data_value_counts is None:
    print("ERROR: ubq_data_value_counts not available. Cannot determine feature columns.")
elif 'TARGET_VARIABLE' not in globals():
    print("ERROR: TARGET_VARIABLE not configured. Run the configuration cell first.")
else:
    # Dynamically determine feature columns from ubq_data_value_counts (all columns except multimer_id)
    feature_cols = [c for c in ubq_data_value_counts.columns if c != 'multimer_id']
    print(f"Using {len(feature_cols)} feature columns from ubq_data_value_counts ({analysis_label})")
    target_col = TARGET_VARIABLE
    print(f"Target variable: {target_col}")

    results = {}
    for sname, df in bandstrength_data_merged.items():
        cur = df.copy()
        # Validate columns
        missing_feats = [c for c in feature_cols if c not in cur.columns]
        if target_col not in cur.columns:
            print(f"Sheet '{sname}': target '{target_col}' missing; skipping.")
            continue
        if missing_feats:
            print(f"Sheet '{sname}': {len(missing_feats)} feature columns missing; continuing with available subset.")
            use_cols = [c for c in feature_cols if c in cur.columns]
        else:
            use_cols = feature_cols
        
        # Prepare X, y (drop rows with NaN in target; fill NaN in features with 0)
        cur = cur.dropna(subset=[target_col])
        if cur.empty:
            print(f"Sheet '{sname}': no rows with non-NaN target; skipping.")
            continue
        X = cur[use_cols].fillna(0)
        y = pd.to_numeric(cur[target_col], errors='coerce')
        valid_mask = ~y.isna()
        X = X.loc[valid_mask]
        y = y.loc[valid_mask]

        # NEW: Show X values for this sheet
        print(f"\nSheet '{sname}': predictor matrix X (showing up to 20 rows)")
        with pd.option_context('display.max_rows', 20, 'display.max_columns', None, 'display.width', 200):
            display(X.head(20))
        
        if len(X) < 10:
            print(f"Sheet '{sname}': insufficient rows ({len(X)}) for train/test split; training on all.")
            rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
            rf.fit(X, y)
            importances = rf.feature_importances_
            fi = pd.DataFrame({
                'feature': X.columns,
                'importance': importances
            }).sort_values('importance', ascending=False)
            results[sname] = {
                'model': rf,
                'feature_importances': fi,
                'train_size': len(X),
                'r2_train': rf.score(X, y)
            }
            print(f"\nSheet '{sname}': trained on all {len(X)} rows (no split). R2_train={results[sname]['r2_train']:.3f}")
            display(fi)
        else:
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
            rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
            rf.fit(X_train, y_train)
            y_pred = rf.predict(X_test)
            r2 = r2_score(y_test, y_pred)
            mse = mean_squared_error(y_test, y_pred)
            importances = rf.feature_importances_
            fi = pd.DataFrame({
                'feature': X.columns,
                'importance': importances
            }).sort_values('importance', ascending=False)
            results[sname] = {
                'model': rf,
                'feature_importances': fi,
                'train_size': len(X_train),
                'test_size': len(X_test),
                'r2_test': r2,
                'mse_test': mse
            }
            print(f"\nSheet '{sname}': RF test metrics -> R2={r2:.3f}, MSE={mse:.5f}, train={len(X_train)}, test={len(X_test)}")
            display(fi)

    rf_overreaction_results = results
    
    # Optional: aggregate features across sheets (no top-10 cap)
    if results:
        agg = []
        for sname, res in results.items():
            fi = res['feature_importances'].copy()
            fi['sheet'] = sname
            agg.append(fi)  # include all features
        agg_df = pd.concat(agg, ignore_index=True)
        print("\nAll features per sheet (aggregated, not capped):")
        with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 200):
            display(agg_df)

Using 63 feature columns from ubq_data_value_counts (Pentamer (Topology-Only))
Target variable: yield
Sheet 'Sheet1': 42 feature columns missing; continuing with available subset.

Sheet 'Sheet1': predictor matrix X (showing up to 20 rows)


,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21
0,2,2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2,2,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2,2,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,2,2,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0
4,2,0,0,0,2,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,2,0,0,0,2,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0
6,2,0,0,0,0,0,0,0,0,2,0,2,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,2,2,0,0,0,2,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,2,2,0,0,0
9,0,0,0,0,0,0,0,0,2,0,0,0,2,0,2,0,0,0,0,0,0



Sheet 'Sheet1': RF test metrics -> R2=-1.018, MSE=0.02086, train=10, test=4


,feature,importance
1,2,0.473949
15,16,0.128772
12,13,0.108470
7,8,0.054968
17,18,0.040985
10,11,0.029891
5,6,0.028052
3,4,0.024701
0,1,0.024623
9,10,0.021595



All features per sheet (aggregated, not capped):


,feature,importance,sheet
0,2,0.473949,Sheet1
1,16,0.128772,Sheet1
2,13,0.108470,Sheet1
3,8,0.054968,Sheet1
4,18,0.040985,Sheet1
5,11,0.029891,Sheet1
6,6,0.028052,Sheet1
7,4,0.024701,Sheet1
8,1,0.024623,Sheet1
9,10,0.021595,Sheet1


# Feature Code → Original JSON Mapping

We produce a dictionary mapping encoded feature columns (`'1'`..`'73'`) back to their original JSON/string values from `ubq_value_to_code_map` to interpret feature importances.

In [457]:
# Build mapping: encoded feature number -> original value (JSON/string)

import json
import pandas as pd

if 'ubq_value_to_code_map' not in globals() or ubq_value_to_code_map is None:
    print("ERROR: ubq_value_to_code_map not available. Run the encoding steps earlier.")
else:
    # Invert the mapping {value: code} -> {code: value}
    code_to_value = {int(code): value for value, code in ubq_value_to_code_map.items()}

    # Dynamically determine feature range from the actual mapping (works for any configuration)
    feature_range = sorted(code_to_value.keys())
    feature_json_map = {str(k): code_to_value[k] for k in feature_range}

    print(f"Feature code → original value mapping ({analysis_label}):")
    print(f"Total features: {len(feature_json_map)}")
    print("\nFirst 10 mappings:")
    for k in feature_range[:10]:
        val_preview = feature_json_map[str(k)][:100] + "..." if len(feature_json_map[str(k)]) > 100 else feature_json_map[str(k)]
        print(f"  {k}: {val_preview}")

    feature_json_df = pd.DataFrame({
        'feature_code': [str(k) for k in feature_range],
        'original_value': [feature_json_map[str(k)] for k in feature_range]
    })
    
    ubq_feature_json_map = feature_json_map


Feature code → original value mapping (Pentamer (Topology-Only)):
Total features: 63

First 10 mappings:
  1: {'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQR...
  2: {'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQR...
  3: {'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQR...
  4: {'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQR...
  5: {'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQR...
  6: {'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQR...
  7: {'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQR...
  8: {'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQR...
  9: {'protein': '1ubq'

In [458]:
# Append original JSON/string to top features

import pandas as pd

# Requires: agg_df (from RF cell) and ubq_feature_json_map (from mapping cell)
if 'agg_df' not in globals() or agg_df is None:
    print("ERROR: 'agg_df' not found. Run the Random Forest cell to build aggregated top features.")
elif 'ubq_feature_json_map' not in globals() or ubq_feature_json_map is None:
    print("ERROR: 'ubq_feature_json_map' not found. Run the feature mapping cell to build code→value map.")
else:
    # Map feature code (string) to original JSON/string value
    mapper = ubq_feature_json_map
    enriched = agg_df.copy()
    # Ensure 'feature' column is string codes
    enriched['feature'] = enriched['feature'].astype(str)
    enriched['original_value'] = enriched['feature'].map(mapper)
    
    print(f"\nFeatures per sheet with original JSON/string ({analysis_label}):")
    
    agg_df_with_values = enriched


Features per sheet with original JSON/string (Pentamer (Topology-Only)):


In [459]:
# Append multimer_ids where each original value appears in ubiquitin history

import pandas as pd

# Requires: agg_df_with_values, ubq_data_clean
if 'agg_df_with_values' not in globals() or agg_df_with_values is None:
    print("ERROR: 'agg_df_with_values' not found. Run the enrichment cell to build it.")
elif 'ubq_data_clean' not in globals() or ubq_data_clean is None:
    print("ERROR: 'ubq_data_clean' not found. Run the ubiquitin cleaning cell first.")
else:
    enriched_ids = agg_df_with_values.copy()
    ubq_df = ubq_data_clean.copy()
    # Ensure we have the multimer_id column
    id_col = 'multimer_id' if 'multimer_id' in ubq_df.columns else None
    if id_col is None:
        print("ERROR: 'multimer_id' column not present in ms4_ubq_clean; cannot lookup.")
    else:
        # Non-ID columns to search for original value occurrences
        search_cols = [c for c in ubq_df.columns if c != id_col]
        # Build list of multimer_ids for each original_value
        def find_ids_for_value(val):
            if pd.isna(val):
                return []
            ids = set()
            for c in search_cols:
                s = ubq_df[c]
                if s.dtype == 'object':
                    mask = s == val
                else:
                    try:
                        vnum = pd.to_numeric(pd.Series([val]), errors='coerce').iloc[0]
                        mask = s == vnum
                    except Exception:
                        mask = pd.Series([False] * len(s), index=s.index)
                ids.update(ubq_df.loc[mask, id_col].astype(str).tolist())
            return sorted(ids)
        
        # NEW: Build list of column names where the original_value appears
        def find_columns_for_value(val):
            if pd.isna(val):
                return []
            cols = []
            for c in search_cols:
                s = ubq_df[c]
                if s.dtype == 'object':
                    has_match = (s == val).any()
                else:
                    try:
                        vnum = pd.to_numeric(pd.Series([val]), errors='coerce').iloc[0]
                        has_match = (s == vnum).any()
                    except Exception:
                        has_match = False
                if has_match:
                    cols.append(c)
            return sorted(cols)
        
        enriched_ids['multimer_ids'] = enriched_ids['original_value'].apply(find_ids_for_value)
        enriched_ids['matching_columns'] = enriched_ids['original_value'].apply(find_columns_for_value)
        print(f"\nFeature importance with original values, matching multimer_ids, and column titles ({analysis_label}):")
        with pd.option_context('display.max_rows', 10, 'display.max_columns', None, 'display.width', 200):
            display(enriched_ids.head(10))
        
        agg_df_with_values_and_ids = enriched_ids


Feature importance with original values, matching multimer_ids, and column titles (Pentamer (Topology-Only)):


,feature,importance,sheet,original_value,multimer_ids,matching_columns
0,2,0.473949,Sheet1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","[Ub5_1, Ub5_10, Ub5_11, Ub5_13, Ub5_2, Ub5_3, Ub5_4, Ub5_5, Ub5_6, Ub5_7, Ub5_8, Ub5_9]","[trimer_deprotection, trimer_formation]"
1,16,0.128772,Sheet1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...",[Ub5_15],"[final_multimer, pentamer_formation]"
2,13,0.108470,Sheet1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...",[Ub5_8],"[final_multimer, pentamer_formation]"
3,8,0.054968,Sheet1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...",[Ub5_6],"[final_multimer, pentamer_formation]"
4,18,0.040985,Sheet1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","[Ub5_17, Ub5_19, Ub5_20]","[tetramer_deprotection, tetramer_formation]"
5,11,0.029891,Sheet1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","[Ub5_10, Ub5_11, Ub5_8]","[tetramer_deprotection, tetramer_formation]"
6,6,0.028052,Sheet1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","[Ub5_6, Ub5_7]","[tetramer_deprotection, tetramer_formation]"
7,4,0.024701,Sheet1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...",[Ub5_1],"[final_multimer, pentamer_formation]"
8,1,0.024623,Sheet1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","[Ub5_1, Ub5_10, Ub5_11, Ub5_13, Ub5_15, Ub5_16, Ub5_17, Ub5_19, Ub5_2, Ub5_20, Ub5_21, Ub5_22, Ub5_26, Ub5_3, Ub5_4,...","[dimer_deprotection, dimer_formation]"
9,10,0.021595,Sheet1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","[Ub5_15, Ub5_16, Ub5_17, Ub5_19, Ub5_20]","[trimer_deprotection, trimer_formation]"


In [460]:
agg_df_with_values_and_ids

,feature,importance,sheet,original_value,multimer_ids,matching_columns
0,2,0.473949,Sheet1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","[Ub5_1, Ub5_10, Ub5_11, Ub5_13, Ub5_2, Ub5_3, Ub5_4, Ub5_5, Ub5_6, Ub5_7, Ub5_8, Ub5_9]","[trimer_deprotection, trimer_formation]"
1,16,0.128772,Sheet1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...",[Ub5_15],"[final_multimer, pentamer_formation]"
2,13,0.108470,Sheet1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...",[Ub5_8],"[final_multimer, pentamer_formation]"
3,8,0.054968,Sheet1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...",[Ub5_6],"[final_multimer, pentamer_formation]"
4,18,0.040985,Sheet1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","[Ub5_17, Ub5_19, Ub5_20]","[tetramer_deprotection, tetramer_formation]"
5,11,0.029891,Sheet1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","[Ub5_10, Ub5_11, Ub5_8]","[tetramer_deprotection, tetramer_formation]"
6,6,0.028052,Sheet1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","[Ub5_6, Ub5_7]","[tetramer_deprotection, tetramer_formation]"
7,4,0.024701,Sheet1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...",[Ub5_1],"[final_multimer, pentamer_formation]"
8,1,0.024623,Sheet1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","[Ub5_1, Ub5_10, Ub5_11, Ub5_13, Ub5_15, Ub5_16, Ub5_17, Ub5_19, Ub5_2, Ub5_20, Ub5_21, Ub5_22, Ub5_26, Ub5_3, Ub5_4,...","[dimer_deprotection, dimer_formation]"
9,10,0.021595,Sheet1,"{'protein': '1ubq', 'chain_number': 1, 'FASTA_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSD...","[Ub5_15, Ub5_16, Ub5_17, Ub5_19, Ub5_20]","[trimer_deprotection, trimer_formation]"
